# фильтрация артефактов ЭМГ

## импорты

In [ ]:
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy import linalg, signal

mne.set_log_level("WARNING")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.grid"] = True


## пути

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_raw_dir = project_root / "data" / "raw"
data_interim_dir = project_root / "data" / "interim"
data_processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "outputs" / "figures"
qc_dir = project_root / "outputs" / "qc"
tables_dir = project_root / "outputs" / "tables"

for folder in [data_interim_dir, data_processed_dir, figures_dir, qc_dir, tables_dir]:
    folder.mkdir(parents=True, exist_ok=True)

fif_path = data_raw_dir / "raw_artifacts_emg.fif"

# пути для ручного запуска
# fif_path = Path(r"/Users/user/Desktop/raw_artifacts_emg.fif")
# fif_path = Path(r"C:\\Users\\user\\Desktop\\raw_artifacts_emg.fif")

recording_name = (
    fif_path.name
    .removesuffix(".fif.gz")
    .removesuffix(".fif")
)
annotations_path = qc_dir / f"{recording_name}-annot.fif"
legacy_annotations_path = qc_dir / f"{recording_name}_annotations.csv"

print("Project root:", project_root)
print("Input FIF:", fif_path)
print("Annotations:", annotations_path)
print("Interim:", data_interim_dir)
print("Processed:", data_processed_dir)
print("Figures:", figures_dir)
print("QC:", qc_dir)
print("Tables:", tables_dir)


## загрузка данных

In [ ]:
if not fif_path.exists():
    raise FileNotFoundError(f"Файл не найден: {fif_path}")

raw_original = mne.io.read_raw_fif(fif_path, preload=True)
raw_original.set_channel_types({
    channel: "emg" for channel in raw_original.ch_names
})
raw_base = raw_original.copy()

# параметры влияют только на отображение, данные остаются в вольтах
emg_browser_kwargs = dict(
    duration=1.0,
    n_channels=8,
    scalings={"emg": 4e-3},
)

# восстанавливаем сохранённую разметку при повторном запуске
if annotations_path.exists():
    saved_annotations = mne.read_annotations(annotations_path)
    raw_base.set_annotations(saved_annotations)
    annotations_source = annotations_path
elif legacy_annotations_path.exists():
    # преобразуем onset старого CSV из времени Unix в секунды записи
    annotations_table = pd.read_csv(legacy_annotations_path)
    epoch = pd.Timestamp("1970-01-01", tz="UTC")
    annotation_onsets = (
        pd.to_datetime(annotations_table["onset"], utc=True) - epoch
    ).dt.total_seconds().to_numpy()
    saved_annotations = mne.Annotations(
        onset=annotation_onsets,
        duration=annotations_table["duration"].to_numpy(),
        description=annotations_table["description"].astype(str).to_numpy(),
        orig_time=None,
    )
    raw_base.set_annotations(saved_annotations)
    annotations_source = legacy_annotations_path
else:
    annotations_source = None

sfreq = raw_base.info["sfreq"]
duration_s = raw_base.n_times / sfreq

print("File:", fif_path.name)
print("Channels:", len(raw_base.ch_names))
print("sfreq:", sfreq)
print("Duration, s:", round(duration_s, 2))
print("First channels:", raw_base.ch_names[:10])
print("Annotations:", len(raw_base.annotations))
print("Annotations loaded from:", annotations_source or "input FIF")


## проверка записи

In [ ]:
channels_table = pd.DataFrame(
    {
        "channel": raw_base.ch_names,
        "type": raw_base.get_channel_types(),
        "bad": [ch in raw_base.info["bads"] for ch in raw_base.ch_names],
    }
)

display(channels_table)
print(f"Sampling rate: {raw_base.info['sfreq']} Hz")
print(f"Duration: {duration_s:.2f} s")
print("Bad channels:", raw_base.info["bads"] or "none")

raw_base.plot(**emg_browser_kwargs)


## сохранение аннотаций

In [ ]:

raw_base.annotations.save(
    annotations_path,
    overwrite=True,
)

print(f"Сохранено аннотаций: {len(raw_base.annotations)}")
print("Файл:", annotations_path)

## выбор каналов

In [ ]:
emg_channels = ["GM R", "GM L", "RF R", "RF L", "TA R", "TA L", "BF R", "BF L"]
right_channels = ["GM R", "RF R", "TA R", "BF R"]
left_channels = ["GM L", "RF L", "TA L", "BF L"]

excluded_channels = ["TA L"]

print("EMG channels:", emg_channels)
print("Right:", right_channels)
print("Left:", left_channels)
print("Excluded:", excluded_channels)


## CAR глобальный

In [ ]:
car_channels = [
    channel for channel in emg_channels
    if channel not in excluded_channels
]

raw_car_global = raw_base.copy()
reference_signal = raw_car_global.get_data(picks=car_channels).mean(axis=0)
channel_indices = [
    raw_car_global.ch_names.index(channel)
    for channel in car_channels
]
raw_car_global._data[channel_indices] -= reference_signal

car_global_path = data_interim_dir / "recording_car_global_raw.fif"
raw_car_global.save(car_global_path, overwrite=True)

print("Saved:", car_global_path)
raw_car_global.plot(**emg_browser_kwargs)

## CAR по сторонам

In [ ]:
raw_car_left_right = raw_base.copy()

for side_channels in [right_channels, left_channels]:
    car_channels = [
        channel for channel in side_channels
        if channel not in excluded_channels
    ]
    reference_signal = raw_car_left_right.get_data(
        picks=car_channels
    ).mean(axis=0)
    channel_indices = [
        raw_car_left_right.ch_names.index(channel)
        for channel in car_channels
    ]
    raw_car_left_right._data[channel_indices] -= reference_signal

car_left_right_path = (
    data_interim_dir / "recording_car_left_right_raw.fif"
)
raw_car_left_right.save(car_left_right_path, overwrite=True)

print("Saved:", car_left_right_path)
raw_car_left_right.plot(**emg_browser_kwargs)

## подготовка методов

In [ ]:
ARTIFACT_LABEL = "artifact"

analysis_channels = [
    channel for channel in emg_channels
    if channel not in excluded_channels
]

missing_channels = [
    channel for channel in analysis_channels
    if channel not in raw_base.ch_names
]
if missing_channels:
    raise ValueError(f"В записи отсутствуют каналы: {missing_channels}")

print("Analysis channels:", analysis_channels)
print("Excluded channels:", excluded_channels)

### маски интервалов

In [ ]:
artifact_mask = np.zeros(raw_base.n_times, dtype=bool)
artifact_intervals = []

for annotation in raw_base.annotations:
    if ARTIFACT_LABEL.lower() not in annotation["description"].lower():
        continue

    onset_from_start = annotation["onset"] - raw_base.first_time
    start = int(raw_base.time_as_index(onset_from_start, use_rounding=True)[0])
    stop = int(raw_base.time_as_index(
        onset_from_start + annotation["duration"], use_rounding=True
    )[0])
    start = max(0, start)
    stop = min(raw_base.n_times, max(start + 1, stop))

    artifact_mask[start:stop] = True
    artifact_intervals.append((start, stop))

if not artifact_mask.any():
    raise ValueError(f"Нет аннотаций с меткой {ARTIFACT_LABEL!r}")

print("Artifact intervals:", len(artifact_intervals))
print("Artifact duration, s:", round(artifact_mask.sum() / sfreq, 3))

## SVD rank-1

In [ ]:
svd_data = raw_base.get_data(picks=analysis_channels)
svd_artifact_data = svd_data[:, artifact_mask]

svd_covariance = np.cov(svd_artifact_data)
left_vectors, svd_singular_values, _ = np.linalg.svd(
    svd_covariance, full_matrices=False
)
artifact_direction = left_vectors[:, :1]  # главная компонента

# строим ортогональный проектор для удаления главной компоненты
svd_projector = (
    np.eye(len(analysis_channels))
    - artifact_direction @ artifact_direction.T
)

# применяем проектор только к размеченным отсчётам
svd_clean_artifact = svd_projector @ svd_artifact_data
raw_svd = raw_base.copy().load_data()
svd_indices = [
    raw_svd.ch_names.index(channel)
    for channel in analysis_channels
]
raw_svd._data[np.ix_(svd_indices, artifact_mask)] = svd_clean_artifact

svd_explained_fraction = (
    svd_singular_values[0] / svd_singular_values.sum()
)
print("First SVD covariance fraction:", round(svd_explained_fraction, 4))
print("Projector symmetric:", np.allclose(svd_projector, svd_projector.T))
print(
    "Projector idempotent:",
    np.allclose(svd_projector @ svd_projector, svd_projector),
)
print("Outside annotations unchanged:", np.array_equal(
    raw_svd.get_data(picks=analysis_channels)[:, ~artifact_mask],
    svd_data[:, ~artifact_mask],
))

## GED

### параметры

In [ ]:
GED_LOWPASS_HZ = 10.0  # частота среза
GED_BASELINE_S = 0.5  # длина фона
GED_GUARD_S = 0.05  # отступ от артефакта
GED_REGULARIZATION = 0.05  # регуляризация ковариации
GED_MIN_POWER_RATIO = 3.0  # порог мощности
GED_APPLY_BEFORE_S = 0.05  # расширение до артефакта
GED_APPLY_AFTER_S = 0.25  # расширение после артефакта
GED_FADE_S = 0.075  # длина плавного края

ged_model_path = qc_dir / "ged_artifact_model_and_masks.npz"
ged_output_path = data_interim_dir / "recording_ged_masked_raw.fif"

print("Low-pass for artifact model, Hz:", GED_LOWPASS_HZ)

### маски применения

In [ ]:
apply_mask = np.zeros(raw_base.n_times, dtype=bool)
normal_mask = np.zeros(raw_base.n_times, dtype=bool)
apply_intervals = []

before = int(round(GED_APPLY_BEFORE_S * sfreq))
after = int(round(GED_APPLY_AFTER_S * sfreq))
for start, stop in artifact_intervals:
    apply_start = max(0, start - before)
    apply_stop = min(raw_base.n_times, stop + after)
    apply_mask[apply_start:apply_stop] = True
    apply_intervals.append((apply_start, apply_stop))

baseline = int(round(GED_BASELINE_S * sfreq))
guard = int(round(GED_GUARD_S * sfreq))
for apply_start, apply_stop in apply_intervals:
    before_start = max(0, apply_start - guard - baseline)
    before_stop = max(0, apply_start - guard)
    after_start = min(raw_base.n_times, apply_stop + guard)
    after_stop = min(raw_base.n_times, apply_stop + guard + baseline)

    normal_mask[before_start:before_stop] = True
    normal_mask[after_start:after_stop] = True

# исключаем область применения GED из чистых интервалов
normal_mask &= ~apply_mask

print("Local normal EMG duration, s:", round(normal_mask.sum() / sfreq, 3))

### модель артефакта

In [ ]:
emg_data = raw_base.get_data(picks=analysis_channels)
lowpass_filter = signal.butter(
    4,
    GED_LOWPASS_HZ,
    btype="lowpass",
    fs=sfreq,
    output="sos",
)
slow_emg_data = signal.sosfiltfilt(lowpass_filter, emg_data, axis=1)

# межканальная ковариация
def covariance(data):
    centered_data = data - data.mean(axis=1, keepdims=True)
    sample_count = centered_data.shape[1]
    return centered_data @ centered_data.T / (sample_count - 1)


artifact_covariance = covariance(slow_emg_data[:, artifact_mask])
normal_covariance = covariance(slow_emg_data[:, normal_mask])
regularization = (
    GED_REGULARIZATION
    * np.trace(normal_covariance)
    / len(analysis_channels)
)
regularized_normal_covariance = (
    normal_covariance
    + regularization * np.eye(len(analysis_channels))
)

power_ratios, spatial_filters = linalg.eigh(
    artifact_covariance,
    regularized_normal_covariance,
)
selected_components = power_ratios > GED_MIN_POWER_RATIO
if not selected_components.any():
    raise ValueError("GED не нашёл компонент, специфичных для артефакта")

ged_filters = spatial_filters[:, selected_components]
component_signals = ged_filters.T @ slow_emg_data

# возвращаем компоненты в пространство каналов для построения модели артефакта
artifact_component_signals = component_signals[:, artifact_mask]
spatial_patterns = (
    slow_emg_data[:, artifact_mask]
    @ artifact_component_signals.T
    @ np.linalg.pinv(
        artifact_component_signals @ artifact_component_signals.T
    )
)
artifact_model = spatial_patterns @ component_signals

eigenvalue_table = pd.DataFrame({
    "artifact / normal power": power_ratios[::-1],
    "remove": selected_components[::-1],
})
display(eigenvalue_table)
display(pd.DataFrame(
    spatial_patterns,
    index=analysis_channels,
    columns=[
        f"component {index + 1}"
        for index in range(selected_components.sum())
    ],
))

### вычитание компоненты

In [ ]:
apply_weight = np.zeros(raw_base.n_times)
fade_samples = int(round(GED_FADE_S * sfreq))

for start, stop in apply_intervals:
    interval_length = stop - start
    edge_length = min(fade_samples, interval_length // 2)
    window = np.ones(interval_length)

    if edge_length:
        ramp = 0.5 - 0.5 * np.cos(np.linspace(0, np.pi, edge_length))
        window[:edge_length] = ramp
        window[-edge_length:] = ramp[::-1]

    apply_weight[start:stop] = window

clean_emg_data = emg_data - artifact_model * apply_weight
raw_ged = raw_base.copy().load_data()
ged_indices = [
    raw_ged.ch_names.index(channel)
    for channel in analysis_channels
]
raw_ged._data[ged_indices] = clean_emg_data

unchanged_outside = np.array_equal(
    raw_ged.get_data(picks=analysis_channels)[:, ~apply_mask],
    emg_data[:, ~apply_mask],
)
print("Outside annotations unchanged:", unchanged_outside)

## проверка сигнала

In [ ]:
records = {
    "original": raw_base,
    "global CAR": raw_car_global,
    "left/right CAR": raw_car_left_right,
    "SVD rank-1 projector": raw_svd,
    "GED slow component": raw_ged,
}

# среднеквадратичная амплитуда
def rms(data, axis=-1):
    return np.sqrt(np.mean(np.square(data), axis=axis))


rms_table = pd.DataFrame({
    name: rms(
        record.get_data(picks=analysis_channels)[:, artifact_mask],
        axis=1,
    )
    for name, record in records.items()
}, index=analysis_channels)
rms_table.index.name = "channel"
display(rms_table)

clean_slow_data = signal.sosfiltfilt(
    lowpass_filter,
    clean_emg_data,
    axis=1,
)
fast_data = emg_data - slow_emg_data
clean_fast_data = clean_emg_data - clean_slow_data

normal_slow_rms = rms(slow_emg_data[:, normal_mask])
artifact_slow_rms = rms(slow_emg_data[:, artifact_mask])
clean_artifact_slow_rms = rms(clean_slow_data[:, artifact_mask])
artifact_fast_rms = rms(fast_data[:, artifact_mask])
clean_artifact_fast_rms = rms(clean_fast_data[:, artifact_mask])

quality = pd.Series({
    "slow RMS: artifact / local normal, before": (
        artifact_slow_rms / normal_slow_rms
    ),
    "slow RMS: artifact / local normal, after": (
        clean_artifact_slow_rms / normal_slow_rms
    ),
    "fast RMS after / before": (
        clean_artifact_fast_rms / artifact_fast_rms
    ),
}, name="value")
display(quality.to_frame())

## визуальное сравнение

In [ ]:
start, stop = artifact_intervals[0]
plot_channel = analysis_channels[0]
context = int(round(0.25 * sfreq))
view_start = max(0, start - context)
view_stop = min(raw_base.n_times, stop + context)
time = raw_base.times[view_start:view_stop]

fig, axes = plt.subplots(len(records), 1, figsize=(14, 11), sharex=True, sharey=True)
for axis, (name, record) in zip(axes, records.items()):
    values = record.get_data(picks=[plot_channel])[0, view_start:view_stop]
    axis.plot(time, values, linewidth=0.9)
    axis.axvspan(raw_base.times[start], raw_base.times[stop - 1], color="tab:orange", alpha=0.15)
    axis.set_title(name)
    axis.set_ylabel("V")
axes[-1].set_xlabel("Time, s")
fig.suptitle(f"{plot_channel}: first artifact interval")
fig.tight_layout()
plt.show()


## сохранение результатов

In [ ]:
raw_ged.save(ged_output_path, overwrite=True)
svd_output_path = data_interim_dir / "recording_svd_k1_masked_raw.fif"
svd_model_path = qc_dir / "svd_k1_model_and_mask.npz"
raw_svd.save(svd_output_path, overwrite=True)

np.savez(
    ged_model_path,
    channels=np.asarray(analysis_channels),
    covariance_artifact=artifact_covariance,
    covariance_normal=normal_covariance,
    eigenvalues=power_ratios,
    ged_filters=ged_filters,
    spatial_patterns=spatial_patterns,
    artifact_mask=artifact_mask,
    apply_mask=apply_mask,
    normal_mask=normal_mask,
    lowpass_hz=GED_LOWPASS_HZ,
)

np.savez(
    svd_model_path,
    channels=np.asarray(analysis_channels),
    covariance=svd_covariance,
    singular_values=svd_singular_values,
    artifact_direction=artifact_direction,
    projector=svd_projector,
    artifact_mask=artifact_mask,
)

output_files = {
    "global CAR": car_global_path,
    "left/right CAR": car_left_right_path,
    "SVD rank-1 projector": svd_output_path,
    "GED slow component": ged_output_path,
}
for name, path in output_files.items():
    print(f"{name}: {path}")
print("GED model and masks:", ged_model_path)
print("SVD model and mask:", svd_model_path)

## сравнение методов

### RMS

In [ ]:
comparison_channels = analysis_channels
outside_mask = normal_mask.copy()  # локальные чистые интервалы
epsilon = np.finfo(float).eps

before_data = raw_base.get_data(picks=comparison_channels)
artifact_rms_before = rms(before_data[:, artifact_mask], axis=1)
normal_rms_before = rms(before_data[:, outside_mask], axis=1)

metric_rows = []
channel_metrics = {}

for method, record in records.items():
    after_data = record.get_data(picks=comparison_channels)
    artifact_rms = rms(after_data[:, artifact_mask], axis=1)
    normal_rms = rms(after_data[:, outside_mask], axis=1)
    values = {
        "Artifact RMS": artifact_rms,
        "Local normal RMS": normal_rms,
        "Artifact RMS ratio": (
            artifact_rms / (artifact_rms_before + epsilon)
        ),
        "Local normal RMS ratio": (
            normal_rms / (normal_rms_before + epsilon)
        ),
    }
    channel_metrics[method] = pd.DataFrame(
        values,
        index=comparison_channels,
    )

    for metric, vector in values.items():
        q1, median, q3 = np.nanpercentile(vector, [25, 50, 75])
        metric_rows.append({
            "method": method,
            "metric": metric,
            "median": median,
            "q1": q1,
            "q3": q3,
        })

metrics_long = pd.DataFrame(metric_rows)

# форматируем медиану и межквартильный интервал
def format_metric(row):
    return f"{row['median']:.3f} [{row['q1']:.3f}–{row['q3']:.3f}]"


metrics_display = metrics_long.assign(
    value=metrics_long.apply(format_metric, axis=1)
).pivot(
    index="metric",
    columns="method",
    values="value",
)
metric_order = list(channel_metrics["original"].columns)
display(metrics_display.reindex(metric_order))

### сравнение до и после

In [ ]:
plot_channel = "TA R"  # канал для графика
method_records = {name: rec for name, rec in records.items() if name != "original"}
if not artifact_intervals:
    raise ValueError("Нет размеченных артефактов для визуализации")

# выбираем ближайшие артефакты для компактного сравнения
if len(artifact_intervals) > 1:
    pair_start = np.argmin([artifact_intervals[i + 1][0] - artifact_intervals[i][1] for i in range(len(artifact_intervals) - 1)])
    shown_intervals = artifact_intervals[pair_start:pair_start + 2]
else:
    shown_intervals = artifact_intervals[:1]
context = int(round(0.35 * sfreq))
view = slice(max(0, shown_intervals[0][0] - context), min(raw_base.n_times, shown_intervals[-1][1] + context))
t = raw_base.times[view]
before_view = raw_base.get_data(picks=[plot_channel])[0, view]
after_views = {name: rec.get_data(picks=[plot_channel])[0, view] for name, rec in method_records.items()}
all_signals = np.concatenate([before_view, *after_views.values()])
signal_limit = 1.05 * np.max(np.abs(all_signals))

fig, axes = plt.subplots(len(method_records), 1, figsize=(16, 3.1 * len(method_records)), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for ax, (method, after_view) in zip(axes, after_views.items()):
    ax.plot(t, before_view, color="#E63946", lw=1.6, alpha=0.95, label="before", zorder=3)
    ax.plot(t, after_view, color="#0066FF", lw=1.35, alpha=0.95, label="after", zorder=4)
    ax.set_ylim(-signal_limit, signal_limit)
    ax.set_ylabel("Amplitude, V")
    ax.set_title(method, loc="left", fontweight="bold")
    ax.grid(alpha=0.18)
    for start, stop in shown_intervals:
        ax.axvspan(raw_base.times[start], raw_base.times[stop - 1], color="#FFB000", alpha=0.22, zorder=1)
axes[0].legend(loc="upper right", frameon=True, ncol=2)
axes[-1].set_xlabel("Time, s")
fig.suptitle(f"{plot_channel}: before/after на размеченных артефактах", y=1.01)
fig.tight_layout()
plt.show()


### отношения RMS

In [ ]:
method_order = list(records)
artifact_heat = pd.DataFrame({m: channel_metrics[m]["Artifact RMS ratio"] for m in method_order}).T
normal_heat = pd.DataFrame({m: channel_metrics[m]["Local normal RMS ratio"] for m in method_order}).T

fig, axes = plt.subplots(1, 2, figsize=(18, 5), constrained_layout=True)
for ax, table, title, cmap, limits in [
    (axes[0], artifact_heat, "Artifact RMS: after / before", "viridis", (0, max(1, np.nanpercentile(artifact_heat, 95)))),
    (axes[1], normal_heat, "Local normal RMS: after / before (идеал = 1)", "coolwarm", (0.5, 1.5)),
]:
    image = ax.imshow(table, aspect="auto", cmap=cmap, vmin=limits[0], vmax=limits[1])
    ax.set_xticks(range(len(table.columns)), table.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(table.index)), table.index)
    ax.set_title(title)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            ax.text(j, i, f"{table.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8, color="black")
    fig.colorbar(image, ax=ax, shrink=0.85)
plt.show()


### межканальные корреляции

In [ ]:
corr_matrices = {
    method: np.corrcoef(record.get_data(picks=comparison_channels)[:, artifact_mask])
    for method, record in records.items()
}
fig, axes = plt.subplots(1, len(corr_matrices), figsize=(4.2 * len(corr_matrices), 4.3), constrained_layout=True)
for ax, (method, matrix) in zip(axes, corr_matrices.items()):
    image = ax.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(method)
    ax.set_xticks(range(len(comparison_channels)), comparison_channels, rotation=90)
    ax.set_yticks(range(len(comparison_channels)), comparison_channels)
fig.colorbar(image, ax=axes, shrink=0.75, label="Pearson r")
fig.suptitle("Межканальная корреляция внутри размеченных артефактов")
plt.show()

# вычитаем исходную матрицу для оценки изменения корреляций
fig, axes = plt.subplots(1, len(corr_matrices) - 1, figsize=(4.2 * (len(corr_matrices) - 1), 4.3), constrained_layout=True)
delta_limit = max(np.max(np.abs(m - corr_matrices["original"])) for name, m in corr_matrices.items() if name != "original")
for ax, (method, matrix) in zip(axes, [(n, m) for n, m in corr_matrices.items() if n != "original"]):
    delta = matrix - corr_matrices["original"]
    image = ax.imshow(delta, cmap="coolwarm", vmin=-delta_limit, vmax=delta_limit)
    ax.set_title(method)
    ax.set_xticks(range(len(comparison_channels)), comparison_channels, rotation=90)
    ax.set_yticks(range(len(comparison_channels)), comparison_channels)
fig.colorbar(image, ax=axes, shrink=0.75, label=r"$\Delta$ Pearson r")
fig.suptitle("Изменение корреляции относительно original")
plt.show()


## просмотр результата

In [ ]:
raw_ged.plot(**emg_browser_kwargs)